## **08 -- Landmark and incident-SMM analyses **

**Place in the framework.** This notebook carries out stage E9 for severe maternal morbidity (SMM) and the returns taken after the
first gate-P4 classification. Each section below starts with a **Framework** line that names the stage, gate, return or criterion it serves.

| Section | Framework | What it produces |
|---|---|---|
| 1 | E1 | Fixed configuration: seed, folds, repeats, bootstrap replicates, reference counts |
| 2 | E7 · E4 / P1 | Predictor dictionary, coverage roles, plausibility limits, missingness indicators; check against the frozen reference counts |
| 3, 3b | E6 · E4 | Populations and labels of each analysis (landmark rule); SMM events by timing class (cohort funnel) |
| 4-6 | E8 · P2 | Patient-grouped nested validation, stacking, Platt recalibration, metrics, patient bootstrap, reports |
| 7 | E9 · C1, C2 | Landmark and incident populations |
| 8 | E9 · C3 | Detection bias of laboratory AKI (tested patients only) |
| 9 | E9 · E3 | Alternative SMM definition (non-acute codes removed) |
| 10 | R4 → E7 · C1, C2 | Extended laboratory set in landmark and incident populations |
| 11 | R2 → E3 · C2 | Escalation of care after T as the timed outcome |
| 12 | E7 · E10 | Race and ethnicity: model without the variable, performance and calibration by group |
| 13 | R4 · C3 | Detection bias for the extended laboratory model |
| 14 | R2 · P3 | Sample size (Riley criteria) for the outcome of return R2 |

Criterion C4 (stability over time) is in notebook 09. The gate-P4 classification itself (D20) is made from the reports of both notebooks.

Framework stages: E1 Governance and configuration · E2 Universe and population · E3 Outcome definition · E4 Count audit ·
E5 Unit of analysis and overlap · E6 Prediction point and window · E7 Predictors · E8 Modeling, validation and metrics ·
E9 Sensitivity and robustness · E10 From classification to action · E11 Reporting. Gates P1-P4; returns R1-R4; gate-P4 criteria C1-C4
(C1 events already present, C2 outcome with time, C3 detection bias, C4 stability over time).

**Purpose.** SMM is an admission-level label and part of the events had begun before the prediction point
(T = admission time + 24 h). This notebook re-runs the main pipeline on populations and labels that remove
morbidity already present at T, using the timing classes of `mimiciv.timing_flags_v3`
(`sql/07_timing_audit/01_timing_flags_v3.sql`):

| Class | Rule (SMM admissions only) |
|---|---|
| `1 prevalent` | a procedure indicator dated before T, **or** ICU admission before T, **or** renal-failure code with laboratory AKI onset at or before T |
| `2 indeterminate` | none of the above, and a procedure indicator on the calendar day that contains T |
| `3 incident (documented)` | none of the above, and a procedure indicator after T **or** renal-failure code with laboratory AKI onset after T |
| `4 untimed` | the rest (diagnosis codes only, no laboratory timing) |

**Analyses** (same pipeline, seed, partitions, tuning, stacking, recalibration and patient-level bootstrap as notebook 02):

| Name | Population | Outcome |
|---|---|---|
| `main` | eligible deliveries | `smm_any` (reference, already reported) |
| `landmark` | eligible deliveries **without** ICU admission before T; SMM admissions of classes 1 and 2 **removed** | `smm_any` |
| `incident_documented` | `landmark` population; SMM admissions of class 4 also **removed** | `smm_any` (only class-3 events remain) |
| `incident_strict` | `landmark` population; SMM admissions not meeting `smm_incident_strict` **removed** | `smm_any` |
| `aki_lab_incident` | eligible deliveries without ICU admission before T and without laboratory AKI at or before T | laboratory AKI with onset after T (code-independent) |

Admissions are removed, never relabelled as non-events. `aki_lab_incident` depends on creatinine being measured after T,
which is itself a care decision; this is stated with its results.

**Data-use rules.** Only aggregate results are printed; files written: `data/outputs_<analysis>.txt` (aggregate text).

**Run order.** Run `sql/07_timing_audit/01_timing_flags_v3.sql` first. Then this notebook top to bottom, `QUICK = True` for a smoke test
(never report QUICK numbers), then `QUICK = False`.

### **1. Libraries, connection and settings**

> **Framework:** E1 Governance and configuration. Fixes the seed, folds, repeats and bootstrap replicates (D1), and the reference counts `EXPECTED` frozen at gate P1.

In [1]:
import os, sys, time, math, warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# notebooks/ -> project root is one level up
sys.path.append(os.path.abspath(os.path.join("..")))
import libraries.database as db_lib
engine = db_lib.get_connect_database()

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

# ----------------------------------------------------------------------------- settings
QUICK        = False         # True = smoke test (small forests, 1 repeat, B = 50). Never report QUICK numbers.
SEED         = 20260101
OUTER_FOLDS  = 5
N_REPEATS    = 3            # repeats of the outer grouped CV (development instability)
INNER_FOLDS  = 3
BOOT         = 1000         # cluster bootstrap replicates for the main analysis
BOOT_SENS    = 300          # replicates for the sensitivity analyses
N_JOBS       = -1
N_TREES      = 300
XGB_TREES    = 200
APPLY_LIMITS = True         # set values outside the plausibility limits of the dictionary to missing
CAT_COLS     = ["ethnicity", "insurance", "language", "admission_type", "admission_location"]
DCA_T        = [0.005, 0.01, 0.02, 0.03, 0.05, 0.10, 0.15, 0.20]
EXPECTED     = {"eligible_deliveries": 19383, "smm": 445, "deaths": 3}   # from the audit counts (Q-F / Q-G)

if QUICK:
    OUTER_FOLDS, N_REPEATS, BOOT, BOOT_SENS = 3, 1, 50, 30
    N_TREES, XGB_TREES = 60, 50
print(f"QUICK={QUICK}  outer={OUTER_FOLDS}x{N_REPEATS}  inner={INNER_FOLDS}  B={BOOT}  trees={N_TREES}")

QUICK=False  outer=5x3  inner=3  B=1000  trees=300


### **2. Load the predictor table and the final dictionary**

> **Framework:** E7 Predictors (dictionary, coverage rule, plausibility limits, missingness indicators) and E4 / P1 (the loaded data are checked against the frozen reference counts; a mismatch prints CHECK).

`predictors_24h_v2` has one row per obstetric hospitalization with the predictors observed in the window
(admission - 24 h, admission + 24 h). `predictor_dictionary_final_v2` assigns every variable its role
(`main model`, `ICU sensitivity only`, `laboratory sensitivity ...`) and its plausibility limits.
Only counts are printed.

In [2]:
df = pd.read_sql("SELECT * FROM mimiciv.predictors_24h_v2", engine)
dic = pd.read_sql("""SELECT variable, model_role_final, missingness_indicator_main, plausible_low, plausible_high,
                            n_eligible_delivery, coverage_pct
                     FROM mimiciv.predictor_dictionary_final_v2""", engine)

for c in ["eligible_24h", "icu_in_24h", "is_delivery_v2", "smm_any", "died_same_admission"]:
    df[c] = df[c].fillna(False).astype(bool)
df["admittime"] = pd.to_datetime(df["admittime"])

VAR_MAIN = dic.loc[dic.model_role_final == "main model", "variable"].tolist()
VAR_ICU  = dic.loc[dic.model_role_final == "ICU sensitivity only", "variable"].tolist()
VAR_LAB  = dic.loc[dic.model_role_final.str.startswith("laboratory sensitivity"), "variable"].tolist()
IND_MAIN = dic.loc[dic.missingness_indicator_main.fillna(False).astype(bool), "variable"].tolist()

print("Dictionary roles ->", {"main": VAR_MAIN, "ICU only": len(VAR_ICU), "lab sensitivity": len(VAR_LAB),
                              "missingness indicator in main model": IND_MAIN})

# Plausibility limits (proposals pending clinical confirmation): out-of-range values become missing
if APPLY_LIMITS:
    n_set = {}
    for _, r in dic.dropna(subset=["plausible_low", "plausible_high"]).iterrows():
        v = r["variable"]
        if v in df.columns:
            bad = (df[v] < r["plausible_low"]) | (df[v] > r["plausible_high"])
            n_set[v] = int(bad.sum())
            df.loc[bad, v] = np.nan
    print("Values set to missing by plausibility limits (count per variable):", n_set)

# Missingness indicators (computed row by row from the raw value: no information crosses patients)
for v in set(IND_MAIN + VAR_LAB + VAR_ICU):
    df[f"{v}_missing"] = df[v].isna().astype(int)
df["icu_in_24h_bin"] = df["icu_in_24h"].astype(int)
df["y"] = df["smm_any"].astype(int)

# Audit checks against the counts already certified (aggregate only)
elig = df[df.eligible_24h & df.is_delivery_v2]
chk = {"eligible_deliveries": len(elig), "smm": int(elig.smm_any.sum()), "deaths": int(elig.died_same_admission.sum())}
for k, v in chk.items():
    print(f"{k}: {v}  (expected {EXPECTED[k]})  [{'OK' if v == EXPECTED[k] else 'CHECK: differs from the audit counts'}]")
print("Eligible any obstetric admissions:", int(df.eligible_24h.sum()),
      "| SMM:", int(df[df.eligible_24h].smm_any.sum()), "| deaths:", int(df[df.eligible_24h].died_same_admission.sum()))

Dictionary roles -> {'main': ['hemoglobin', 'creatinine'], 'ICU only': 14, 'lab sensitivity': 11, 'missingness indicator in main model': ['creatinine']}
Values set to missing by plausibility limits (count per variable): {'hemoglobin': 0, 'hematocrit': 0, 'creatinine': 1}
eligible_deliveries: 19383  (expected 19383)  [OK]
smm: 445  (expected 445)  [OK]
deaths: 3  (expected 3)  [OK]
Eligible any obstetric admissions: 23625 | SMM: 814 | deaths: 5


In [3]:
# Timing classes (one row per eligible delivery hospitalization)
tf = pd.read_sql("""SELECT hadm_id, smm_timing_class, smm_incident_strict, aki_prevalent, aki_incident
                     FROM mimiciv.timing_flags_v3""", engine)
for col in ["smm_incident_strict", "aki_prevalent", "aki_incident"]:
    tf[col] = tf[col].fillna(False).astype(bool)
df = df.drop(columns=[x for x in tf.columns if x != "hadm_id" and x in df.columns]).merge(tf, on="hadm_id", how="left")
df["smm_timing_class"] = df["smm_timing_class"].fillna("not an eligible delivery")
for col in ["smm_incident_strict", "aki_prevalent", "aki_incident"]:
    df[col] = df[col].fillna(False).astype(bool)
df["y_aki"] = df["aki_incident"].astype(int)
n_tf = int((df.eligible_24h & df.is_delivery_v2 & (df.smm_timing_class != "not an eligible delivery")).sum())
print(f"Eligible deliveries with timing flags: {n_tf} (expected {EXPECTED['eligible_deliveries']}) "
      f"[{'OK' if n_tf == EXPECTED['eligible_deliveries'] else 'CHECK'}]")

Eligible deliveries with timing flags: 19383 (expected 19383) [OK]


### **3. Analysis datasets and feature sets**

> **Framework:** E6 Prediction point and window. Each specification applies the eligibility and landmark rules (D11, D12) to define its population and label; E9 uses these specifications.

Each analysis is a (population, feature set, label) specification:

| Name | Population | Features |
|---|---|---|
| `main` | Eligible delivery hospitalizations | Main-model variables of the dictionary (hemoglobin, creatinine), creatinine missingness indicator, age, categorical admission variables |
| `first_admission` | Index (first) delivery hospitalization of each patient, if eligible | as `main` |
| `any_obstetric` | Eligible obstetric hospitalizations (delivery or not) | as `main` |
| `labs_extended` | as `main` | `main` + hematocrit and the other laboratory variables, each with a missingness indicator |
| `icu_vars` | as `main` | `main` + the ICU-only variables (each with a missingness indicator) + `icu_in_24h` |
| `label_ge2` | as `main` | as `main`, outcome = two or more SMM indicators (`smm_ge2`) |
| `no_icu_24h` | Eligible deliveries **without** an ICU admission in the first 24 h | as `main` |
| `icd10_only` | Eligible deliveries whose diagnoses are all ICD-10-CM (no ICD-9-CM diagnosis) | as `main` |

The table above is that of notebook 02. This notebook uses `main` (reference) and the four new specifications of the next cell: `landmark`, `incident_documented`, `incident_strict` and `aki_lab_incident`.

In [4]:
def make_spec(name):
    d = df[df.eligible_24h & df.is_delivery_v2].copy()
    num = ["age_at_admission"] + VAR_MAIN
    binc = [f"{v}_missing" for v in IND_MAIN]
    label = "y"
    prevalent_or_indet = d.smm_timing_class.isin(["1 prevalent", "2 indeterminate"])
    if name == "main":
        pass
    elif name in ("landmark", "incident_documented", "incident_strict"):
        d = d[~d.icu_in_24h & ~prevalent_or_indet]
        if name == "incident_documented":
            d = d[~(d.y.astype(bool) & (d.smm_timing_class == "4 untimed"))]
        if name == "incident_strict":
            d = d[~(d.y.astype(bool) & ~d.smm_incident_strict)]
    elif name == "aki_lab_incident":
        d = d[~d.icu_in_24h & ~d.aki_prevalent]
        label = "y_aki"
    else:
        raise ValueError(name)
    num  = [c for c in dict.fromkeys(num) if c in d.columns]
    binc = [c for c in dict.fromkeys(binc) if c in d.columns and d[c].nunique() > 1]
    cat  = [c for c in CAT_COLS if c in d.columns]
    d = d.reset_index(drop=True)
    return dict(name=name, data=d, num=num, bin=binc, cat=cat, label=label)

ANALYSES = ["landmark", "incident_documented", "incident_strict", "aki_lab_incident"]
for nm in ["main"] + ANALYSES:
    sp = make_spec(nm)
    d = sp["data"]
    print(f"{nm:20s} admissions={len(d):6d}  patients={d.subject_id.nunique():6d}  events={int(d[sp['label']].sum()):4d}  "
          f"prevalence={d[sp['label']].mean():.4f}")

main                 admissions= 19383  patients= 13710  events= 445  prevalence=0.0230
landmark             admissions= 19198  patients= 13584  events= 285  prevalence=0.0148
incident_documented  admissions= 19027  patients= 13479  events= 114  prevalence=0.0060
incident_strict      admissions= 18996  patients= 13453  events=  83  prevalence=0.0044
aki_lab_incident     admissions= 19175  patients= 13565  events= 202  prevalence=0.0105


### **3b. SMM admissions by timing class (aggregates only)**

> **Framework:** E4 Count audit. SMM events by timing class relative to T; these counts feed the cohort funnel.

This table shows how many SMM events can be placed in time relative to T; it goes to the manuscript.

In [5]:
e = df[df.eligible_24h & df.is_delivery_v2]
tab = (e[e.y == 1].groupby("smm_timing_class").size().rename("SMM admissions").to_frame())
tab["% of SMM"] = (100 * tab["SMM admissions"] / tab["SMM admissions"].sum()).round(1)
print(tab.to_string())
print(f"\nStrict incident events: {int((e.y.astype(bool) & e.smm_incident_strict).sum())}")
print(f"Laboratory AKI at or before T: {int(e.aki_prevalent.sum())};  after T: {int(e.aki_incident.sum())} "
      f"(of which with an SMM code: {int((e.aki_incident & e.y.astype(bool)).sum())})")
print(f"ICU admission before T: {int(e.icu_in_24h.sum())} admissions ({int((e.icu_in_24h & e.y.astype(bool)).sum())} with SMM)")

                         SMM admissions  % of SMM
smm_timing_class                                 
1 prevalent                         150      33.7
2 indeterminate                      10       2.2
3 incident (documented)             114      25.6
4 untimed                           171      38.4

Strict incident events: 83
Laboratory AKI at or before T: 110;  after T: 217 (of which with an SMM code: 113)
ICU admission before T: 115 admissions (90 with SMM)


### **4. Core functions (preprocessing, learners, nested tuning, stacking, calibration)**

> **Framework:** E8 Modeling, validation and metrics. Everything that learns from data stays inside the training part of each split; the assertion that no patient is on both sides is criterion 4 of gate P2.

Everything that learns from data is inside the training portion of each split:

* `Winsor` clips each numeric column to the 0.5th-99.5th percentiles *of the training data*.
* The median imputer, the scaler and the one-hot encoder are fitted inside a `Pipeline`, once per fit.
* Hyperparameters are chosen by **inner patient-grouped CV** (average precision of the pooled out-of-fold predictions).
* The meta-learner of the ensemble is trained on the **out-of-fold** predictions of the base learners (never on in-sample predictions).
* The Platt calibrator is trained on out-of-fold predictions of the training patients; the outer test patients are only scored.

In [6]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def expit(z):
    return 1.0 / (1.0 + np.exp(-z))

class Winsor(BaseEstimator, TransformerMixin):
    """Clip each column to training percentiles (fit on the training data only)."""
    def __init__(self, lo=0.5, hi=99.5):
        self.lo = lo; self.hi = hi
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lo_ = np.nanpercentile(X, self.lo, axis=0)
        self.hi_ = np.nanpercentile(X, self.hi, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, float), self.lo_, self.hi_)

def make_prep(num, binc, cat):
    parts = [("num", Pipeline([("w", Winsor()),
                               ("imp", SimpleImputer(strategy="median", keep_empty_features=True)),
                               ("sc", StandardScaler())]), num)]
    if binc:
        parts.append(("bin", "passthrough", binc))
    if cat:
        parts.append(("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                                       ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                            min_frequency=0.01, sparse_output=False))]), cat))
    return ColumnTransformer(parts)

def prep_frame(d, num, binc, cat):
    X = d[num + binc + cat].copy()
    for c in cat:
        X[c] = X[c].astype(object).where(X[c].notna(), np.nan)
    return X

# Small prespecified grids (class weights are options; no resampling)
GRID = {
    # The grids cover values beyond those most often chosen in preliminary runs, so that the selected value
    # is not at the edge of the grid (ET depth, RF depth and XGB depth; class weights remain options).
    "LR":  [dict(C=C, cw=cw) for C in (0.01, 0.1, 1.0) for cw in (None, "balanced")],
    "RF":  [dict(depth=dp, cw=cw) for dp in (2, 4, 8) for cw in (None, "balanced_subsample")],
    "ET":  [dict(depth=dp, cw=cw) for dp in (4, 8, 12) for cw in (None, "balanced_subsample")],
    "XGB": [dict(depth=dp, spw=sp) for dp in (1, 2, 4) for sp in (1, "sqrt")],
}
MODELS = ["LR", "RF", "ET", "XGB", "Stack", "Stack_raw"]
BOOT_MODELS = ["LR", "RF", "ET", "XGB", "Stack"]
PAIRS = [("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "LR"), ("Stack", "XGB")]

def cfg_str(c):
    return ",".join(f"{k}={v}" for k, v in c.items())

def make_clf(name, cfg, ytr, seed):
    if name == "LR":
        return LogisticRegression(C=cfg["C"], class_weight=cfg["cw"], max_iter=3000)
    if name == "RF":
        return RandomForestClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                      class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "ET":
        return ExtraTreesClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                    class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "XGB":
        pos = max(int(ytr.sum()), 1); neg = max(len(ytr) - pos, 1)
        spw = math.sqrt(neg / pos) if cfg["spw"] == "sqrt" else 1.0
        return XGBClassifier(n_estimators=XGB_TREES, learning_rate=0.05, max_depth=cfg["depth"], subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=5, scale_pos_weight=spw, eval_metric="logloss",
                             tree_method="hist", n_jobs=N_JOBS if N_JOBS and N_JOBS > 0 else 0, random_state=seed)
    raise ValueError(name)

def fit_predict(name, cfg, prep, Xtr, ytr, Xte, seed):
    pipe = Pipeline([("prep", clone(prep)), ("clf", make_clf(name, cfg, ytr, seed))])
    pipe.fit(Xtr, ytr)
    return pipe.predict_proba(Xte)[:, 1]

def platt(o, t, ytr):
    c = LogisticRegression(penalty=None, max_iter=1000).fit(logit(o).reshape(-1, 1), ytr)
    return c.predict_proba(logit(t).reshape(-1, 1))[:, 1]

def tune_stack_calibrate(Xtr, ytr, gtr, Xte, prep, seed):
    """Inside one outer training set: tune, build out-of-fold predictions, stack, calibrate; score the outer test set."""
    inner = list(StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed).split(Xtr, ytr, gtr))
    names = list(GRID)
    chosen, oof, pred_te = {}, {}, {}
    for name in names:
        best = None
        for cfg in GRID[name]:
            o = np.zeros(len(ytr))
            for tr, va in inner:
                o[va] = fit_predict(name, cfg, prep, Xtr.iloc[tr], ytr[tr], Xtr.iloc[va], seed)
            ap = average_precision_score(ytr, o)
            if best is None or ap > best[0]:
                best = (ap, cfg, o)
        chosen[name] = best[1]; oof[name] = best[2]
        pred_te[name] = fit_predict(name, best[1], prep, Xtr, ytr, Xte, seed)
    # explicit stacking: logistic meta-learner on out-of-fold base predictions (C = 0.1 as in the original framework)
    Z_oof = np.column_stack([logit(oof[n]) for n in names])
    Z_te = np.column_stack([logit(pred_te[n]) for n in names])
    meta = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof, ytr)
    stack_te_raw = meta.predict_proba(Z_te)[:, 1]
    # cross-fitted meta out-of-fold predictions -> calibrator never sees the predictions of its own training rows
    mo = np.zeros(len(ytr))
    for tr, va in StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed + 1).split(Z_oof, ytr, gtr):
        m = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof[tr], ytr[tr])
        mo[va] = m.predict_proba(Z_oof[va])[:, 1]
    out = {n: platt(oof[n], pred_te[n], ytr) for n in names}
    out["Stack"] = platt(mo, stack_te_raw, ytr)
    out["Stack_raw"] = stack_te_raw
    return out, chosen

def run_cv(spec, label=""):
    d = spec["data"]
    X = prep_frame(d, spec["num"], spec["bin"], spec["cat"])
    y = d[spec["label"]].values.astype(int)
    g = d["subject_id"].values
    prep = make_prep(spec["num"], spec["bin"], spec["cat"])
    n = len(y)
    preds = {m: np.full((N_REPEATS, n), np.nan) for m in MODELS}
    chosen_log = []
    split_log = []
    t0 = time.time(); done = 0; total = N_REPEATS * OUTER_FOLDS
    for r in range(N_REPEATS):
        outer = StratifiedGroupKFold(OUTER_FOLDS, shuffle=True, random_state=SEED + r)
        for k, (tr, te) in enumerate(outer.split(X, y, g)):
            assert len(set(g[tr]) & set(g[te])) == 0, "patient present in training and test"
            split_log.append(dict(repeat=r + 1, fold=k + 1,
                                  train_adm=len(tr), train_pat=len(set(g[tr])), train_events=int(y[tr].sum()),
                                  train_event_pat=len(set(g[tr][y[tr] == 1])),
                                  test_adm=len(te), test_pat=len(set(g[te])), test_events=int(y[te].sum()),
                                  test_event_pat=len(set(g[te][y[te] == 1])), shared_patients=0))
            out, chosen = tune_stack_calibrate(X.iloc[tr], y[tr], g[tr], X.iloc[te], prep, SEED + 100 * r + k)
            for m in MODELS:
                preds[m][r, te] = out[m]
            for nme, cfg in chosen.items():
                chosen_log.append((nme, cfg_str(cfg)))
            done += 1
            if done == 1:
                el = time.time() - t0
                print(f"[{spec['name']}] first outer fold took {el/60:.1f} min -> projected total about {el*total/60:.0f} min")
    print(f"[{spec['name']}] cross-validation finished in {(time.time()-t0)/60:.1f} min; "
          f"every admission scored exactly once per repeat: {bool(np.isfinite(preds['Stack']).all())}")
    return dict(spec=spec, y=y, g=g, preds=preds, chosen=chosen_log, X=X, prep=prep, splits=pd.DataFrame(split_log))

### **5. Metrics, cluster bootstrap and paired differences**

> **Framework:** E8 Modeling, validation and metrics. Discrimination, calibration, clinical usefulness and patient-level bootstrap intervals.

All metrics are computed on the pooled out-of-fold predictions of each repeat and then averaged over repeats.
The bootstrap resamples **patients** (all admissions of a patient move together) and recomputes every metric with
patient-count weights, so intervals reflect the repeated-admission structure. Differences against logistic
regression use the **same** bootstrap samples (paired).

In [7]:
def citl(y, z, w):
    """Calibration-in-the-large: intercept with the slope fixed at 1 (weighted Newton)."""
    a = 0.0
    for _ in range(40):
        pr = expit(z + a)
        g = np.sum(w * (y - pr)); h = np.sum(w * pr * (1 - pr))
        if h < 1e-12:
            break
        step = g / h; a += step
        if abs(step) < 1e-9:
            break
    return a

def calib_slope(y, z, w):
    """Calibration slope (weighted logistic recalibration, Newton)."""
    a, b = 0.0, 1.0
    for _ in range(40):
        pr = expit(a + b * z)
        r = w * (y - pr); v = w * pr * (1 - pr)
        gr = np.array([r.sum(), (r * z).sum()])
        H = np.array([[v.sum(), (v * z).sum()], [(v * z).sum(), (v * z * z).sum()]])
        try:
            dlt = np.linalg.solve(H, gr)
        except np.linalg.LinAlgError:
            break
        a += dlt[0]; b += dlt[1]
        if np.abs(dlt).max() < 1e-9:
            break
    return b

def top_threshold(p, w, k):
    order = np.argsort(-p)
    cw = np.cumsum(w[order])
    idx = int(np.searchsorted(cw, k * cw[-1]))
    return p[order][min(idx, len(p) - 1)]

def metrics_one(y, p, w):
    z = logit(p)
    m = {}
    m["auroc"] = roc_auc_score(y, p, sample_weight=w)
    m["auprc"] = average_precision_score(y, p, sample_weight=w)
    m["brier"] = float(np.average((p - y) ** 2, weights=w))
    m["slope"] = calib_slope(y, z, w)
    m["citl"] = citl(y, z, w)
    m["oe"] = float(np.sum(w * y) / np.sum(w * p))
    edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
    bins = np.searchsorted(edges, p)
    W = w.sum(); ece = 0.0
    for b in range(10):
        mk = bins == b; wb = w[mk].sum()
        if wb > 0:
            ece += wb / W * abs(np.sum(w[mk] * y[mk]) / wb - np.sum(w[mk] * p[mk]) / wb)
    m["ece"] = ece
    ev = np.sum(w * y)
    for k in (0.05, 0.10):
        sel = p >= top_threshold(p, w, k)
        m[f"sens_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / ev
        m[f"ppv_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / np.sum(w[sel])
    for pt in DCA_T:
        sel = p >= pt
        tp = np.sum(w[sel] * y[sel]); fp = np.sum(w[sel] * (1 - y[sel]))
        m[f"nb_{pt}"] = tp / W - fp / W * pt / (1 - pt)
    return m

def all_metrics(y, preds, w, models, strata):
    """Metrics for each model, averaged over repeats. Returns dict[(model, metric)] and the per-repeat list."""
    per = []
    for r in range(N_REPEATS):
        cur = {}
        for mname in models:
            for k, v in metrics_one(y, preds[mname][r], w).items():
                cur[(mname, k)] = v
        prev = np.sum(w * y) / w.sum()
        for pt in DCA_T:
            cur[("TreatAll", f"nb_{pt}")] = prev - (1 - prev) * pt / (1 - pt)
        for sname, mask in strata.items():
            for mname in ("LR", "Stack"):
                if mname not in models:
                    continue
                wm = w[mask]; ym = y[mask]; pm = preds[mname][r][mask]
                if np.sum(wm * ym) >= 1 and np.sum(wm * (1 - ym)) >= 1:
                    cur[(mname, f"auroc_{sname}")] = roc_auc_score(ym, pm, sample_weight=wm)
                    cur[(mname, f"auprc_{sname}")] = average_precision_score(ym, pm, sample_weight=wm)
                else:
                    cur[(mname, f"auroc_{sname}")] = np.nan
                    cur[(mname, f"auprc_{sname}")] = np.nan
        per.append(cur)
    keys = per[0].keys()
    mean = {k: float(np.nanmean([c[k] for c in per])) if not all(np.isnan(c[k]) for c in per) else np.nan for k in keys}
    return mean, per

def bootstrap(res, B, strata, seed):
    y, g, preds = res["y"], res["g"], res["preds"]
    ug, inv = np.unique(g, return_inverse=True)
    G = len(ug)
    rng = np.random.default_rng(seed)
    out = {}
    t0 = time.time()
    for b in range(B):
        cnt = np.bincount(rng.integers(0, G, G), minlength=G)
        w = cnt[inv].astype(float)
        if np.sum(w * y) < 1:
            continue
        mean, _ = all_metrics(y.astype(float), preds, w, BOOT_MODELS, strata)
        for k, v in mean.items():
            out.setdefault(k, []).append(v)
        for a_, b_ in PAIRS:            # paired differences (same bootstrap sample)
            for met in ("auroc", "auprc", "brier") + tuple(f"nb_{pt}" for pt in DCA_T):
                out.setdefault((f"{a_}-{b_}", met), []).append(mean[(a_, met)] - mean[(b_, met)])
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return {k: np.array(v) for k, v in out.items()}

### **6. Reporting function**

> **Framework:** E8 and E11 Reporting. Aggregate reports only, saved as `data/outputs_<analysis>.txt`.

Prints aggregate tables only: discrimination, calibration, paired differences against logistic regression, the share of
events captured in the top 5 % / 10 % of predicted risk, net benefit, development instability across repeats and the
hyperparameters chosen in the inner loops.

In [8]:
def _ci(boot, key, d=3):
    a = boot.get(key)
    if a is None or len(a) < 20:
        return ""
    lo, hi = np.nanpercentile(a, [2.5, 97.5])
    return f" ({lo:.{d}f}-{hi:.{d}f})"

def decile_table(res, models=("LR", "Stack")):
    y = res["y"].astype(float)
    out = None
    for m in models:
        acc = []
        for r in range(N_REPEATS):
            p = res["preds"][m][r]
            edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
            b = np.searchsorted(edges, p)
            acc.append(pd.DataFrame({"b": b, "p": p, "y": y}).groupby("b").agg(n=("y", "size"), pred=("p", "mean"), obs=("y", "mean")))
        mean = sum(acc) / len(acc)
        t = pd.DataFrame({"decile": mean.index + 1, "n": mean["n"].round(0).astype(int)}) if out is None else out
        t[f"{m}_pred"] = mean["pred"].round(4).values
        t[f"{m}_obs"] = mean["obs"].round(4).values
        out = t
    return out

def plot_calibration(res):
    import matplotlib.pyplot as plt
    t = decile_table(res)
    fig, ax = plt.subplots(figsize=(5, 5))
    mx = float(max(t["LR_pred"].max(), t["LR_obs"].max(), t["Stack_pred"].max(), t["Stack_obs"].max())) * 1.1
    ax.plot([0, mx], [0, mx], color="grey", lw=1, ls="--", label="ideal")
    ax.plot(t["LR_pred"], t["LR_obs"], "o-", label="Logistic regression")
    ax.plot(t["Stack_pred"], t["Stack_obs"], "s-", label="Stacked ensemble")
    ax.set_xlabel("Mean predicted risk (decile)"); ax.set_ylabel("Observed SMM rate")
    ax.set_title(f"Calibration by decile -- {res['spec']['name']}"); ax.legend(); plt.show()   # displayed only, not saved

def emit(name, fn):
    """Print the aggregate report and also keep it as a text file (aggregates only) so it can be shared without truncation."""
    import io, contextlib
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        fn()
    text = buf.getvalue()
    print(text)
    path = f"../data/outputs_{name}.txt"
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(text)
    print(f"[aggregate report saved as data/{os.path.basename(path)}  ({len(text)} characters)]")

def run_analysis(name, B):
    spec = make_spec(name)
    res = run_cv(spec)
    d = spec["data"]
    strata = {}
    if name == "main":
        strata = {"icu1": d["icu_in_24h"].values.astype(bool), "icu0": ~d["icu_in_24h"].values.astype(bool)}
    y = res["y"].astype(float)
    res["point"], res["per"] = all_metrics(y, res["preds"], np.ones(len(y)), MODELS + [], strata)
    res["boot"] = bootstrap(res, B, strata, SEED + 7)
    res["strata"] = strata
    return res

def report(res):
    sp = res["spec"]; d = sp["data"]; y = res["y"]
    pt, per, bt = res["point"], res["per"], res["boot"]
    n, ev, npat = len(y), int(y.sum()), d.subject_id.nunique()
    print("=" * 110)
    print(f"ANALYSIS {sp['name']}: {n} admissions, {npat} patients, {ev} events ({100*ev/n:.2f}%), "
          f"numeric={len(sp['num'])}, binary={len(sp['bin'])}, categorical={len(sp['cat'])}")
    print(f"Repeats={N_REPEATS}, outer folds={OUTER_FOLDS}, inner folds={INNER_FOLDS}, bootstrap B={len(next(iter(bt.values()))) if bt else 0}, QUICK={QUICK}")
    print("=" * 110)

    sl = res["splits"]
    print("\n[0] Patients per outer partition (repeat 1 shown; every split verified to share no patient between training and test)")
    print(sl[sl.repeat == 1].drop(columns=["repeat"]).to_string(index=False))
    print(f"    Across all {len(sl)} splits: test patients per fold {sl.test_pat.min()}-{sl.test_pat.max()}, "
          f"event-bearing test patients per fold {sl.test_event_pat.min()}-{sl.test_event_pat.max()}, "
          f"patients present on both sides: {int(sl.shared_patients.sum())}")

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        r["sd_auroc_repeats"] = f"{np.std([c[(m,'auroc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        r["sd_auprc_repeats"] = f"{np.std([c[(m,'auprc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        rows.append(r)
    print("\n[1] Discrimination and overall accuracy (mean over repeats; 95% cluster-bootstrap CI)")
    print(f"    prevalence = {y.mean():.4f}  (AUPRC of a non-informative model equals the prevalence)")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("slope", 2), ("citl", 2), ("oe", 2), ("ece", 4)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        rows.append(r)
    print("\n[2] Calibration (slope ideal 1; calibration-in-the-large ideal 0; O/E ideal 1; ECE with 10 quantile bins)")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for a_, b_ in PAIRS:
        r = {"comparison (first minus second)": f"{a_} - {b_}"}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(a_, met)] - pt[(b_, met)]:+.{dd}f}" + _ci(bt, (f"{a_}-{b_}", met), dd)
        rows.append(r)
    print("\n[3] Paired differences (same bootstrap samples); the last row compares the ensemble with XGBoost")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for m in BOOT_MODELS:
        rows.append({"model": m,
                     "sens_top5%": f"{pt[(m,'sens_top5')]:.3f}" + _ci(bt, (m, "sens_top5")),
                     "ppv_top5%": f"{pt[(m,'ppv_top5')]:.3f}",
                     "sens_top10%": f"{pt[(m,'sens_top10')]:.3f}" + _ci(bt, (m, "sens_top10")),
                     "ppv_top10%": f"{pt[(m,'ppv_top10')]:.3f}"})
    print("\n[4] Events captured among the 5% / 10% of admissions with the highest predicted risk")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for m in BOOT_MODELS + ["TreatAll"]:
        rows.append({"strategy": m, **{f"pt={t:g}": round(pt[(m, f'nb_{t}')], 5) for t in DCA_T}})
    rows.append({"strategy": "TreatNone", **{f"pt={t:g}": 0.0 for t in DCA_T}})
    print("\n[5] Net benefit (decision curve) at the stated threshold probabilities")
    print(pd.DataFrame(rows).to_string(index=False))
    rows = []
    for a_, b_ in [("Stack", "LR"), ("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "XGB")]:
        rows.append({"comparison": f"{a_} - {b_}", **{f"pt={t:g}": f"{pt[(a_,f'nb_{t}')]-pt[(b_,f'nb_{t}')]:+.5f}" + _ci(bt, (f"{a_}-{b_}", f"nb_{t}"), 5) for t in DCA_T[:6]}})
    print("    Paired net-benefit differences:")
    print(pd.DataFrame(rows).to_string(index=False))

    if res["strata"]:
        rows = []
        for sname, lab in [("icu1", "ICU admission within 24 h"), ("icu0", "no ICU within 24 h")]:
            mk = res["strata"][sname]
            r = {"stratum": lab, "n": int(mk.sum()), "events": int(y[mk].sum())}
            for m in ("LR", "Stack"):
                r[f"{m}_auroc"] = (f"{pt[(m, 'auroc_' + sname)]:.3f}" + _ci(bt, (m, "auroc_" + sname))) if not np.isnan(pt[(m, 'auroc_' + sname)]) else "n/a"
                r[f"{m}_auprc"] = (f"{pt[(m, 'auprc_' + sname)]:.3f}") if not np.isnan(pt[(m, 'auprc_' + sname)]) else "n/a"
            rows.append(r)
        print("\n[6] Performance by ICU admission within the first 24 h (same predictions, stratified evaluation)")
        print(pd.DataFrame(rows).to_string(index=False))

    print("\n[8] Calibration by decile of predicted risk (outer-test predictions; mean over repeats)")
    print(decile_table(res).to_string(index=False))

    ch = pd.DataFrame(res["chosen"], columns=["learner", "config"])
    tab = ch.value_counts().rename("outer_fits").reset_index().sort_values(["learner", "outer_fits"], ascending=[True, False])
    print(f"\n[7] Hyperparameters selected in the inner loops (out of {N_REPEATS*OUTER_FOLDS} outer fits per learner)")
    print(tab.to_string(index=False))

### **7. Run the analyses**

> **Framework:** E9 Sensitivity and robustness · gate-P4 criteria C1 (`landmark`) and C2 (`incident_documented`, `incident_strict`, `aki_lab_incident`).

Each report has the same layout as notebook 02 (discrimination with patient-level bootstrap intervals, calibration, paired differences
including ensemble minus logistic regression and ensemble minus XGBoost, top 5 % / 10 %, net benefit, hyperparameters).
Analyses with fewer than 40 events are run but flagged as unstable.
Set `RUN_MAIN_ANALYSES = False` to skip these four analyses (for example, when they already ran and only section 8 is needed).

In [9]:
RUN_MAIN_ANALYSES = True   # set to False to run only section 8 without repeating the four analyses above
results = {}
if RUN_MAIN_ANALYSES:
    for nm in ANALYSES:
        sp = make_spec(nm)
        ev = int(sp["data"][sp["label"]].sum())
        if ev < 40:
            print(f"WARNING: {nm} has only {ev} events; estimates will be unstable (report as exploratory).")
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

[landmark] first outer fold took 0.6 min -> projected total about 10 min
[landmark] cross-validation finished in 10.1 min; every admission scored exactly once per repeat: True
  bootstrap: 5 replicates took 1 s -> projected 1.0 min for B=300
ANALYSIS landmark: 19198 admissions, 13584 patients, 285 events (1.48%), numeric=3, binary=1, categorical=5
Repeats=3, outer folds=5, inner folds=3, bootstrap B=300, QUICK=False

[0] Patients per outer partition (repeat 1 shown; every split verified to share no patient between training and test)
 fold  train_adm  train_pat  train_events  train_event_pat  test_adm  test_pat  test_events  test_event_pat  shared_patients
    1      15400      10862           229              227      3798      2722           56              56                0
    2      15364      10868           229              228      3834      2716           56              55                0
    3      15381      10869           223              221      3817      2715        

### **8. Sensitivity of `aki_lab_incident`: detection bias and shared variable**

> **Framework:** E9 · gate-P4 criterion C3 (detection bias) for the main model.

Laboratory AKI after T can only be detected if creatinine is measured after T, and patients whose creatinine was already measured in the
first 24 h are more likely to be measured again. The predictor (creatinine up to 24 h) and the outcome (later creatinine) are also the same
variable. Two analyses separate these effects:

| Analysis | Population | Predictors |
|---|---|---|
| `aki_lab_incident_tested` | `aki_lab_incident` population restricted to admissions **with creatinine measured in the first 24 h** | as `main` (the creatinine missingness indicator is constant here and drops out) |
| `aki_lab_incident_tested_nocreat` | same as above | as `main` **without the creatinine value** (age, hemoglobin, administrative variables) |

Reading: if the AUROC of `aki_lab_incident_tested` stays close to that of `aki_lab_incident`, the result is not driven by who was tested.
The difference between the two analyses of this section shows how much of the discrimination comes from the creatinine value itself.
Reports are saved as `data/outputs_aki_lab_incident_tested.txt` and `data/outputs_aki_lab_incident_tested_nocreat.txt`.

In [10]:
CREAT = [v for v in VAR_MAIN if "creat" in v.lower()]
assert len(CREAT) == 1, f"creatinine variable not found in VAR_MAIN: {VAR_MAIN}"
CREAT = CREAT[0]
_make_spec_base = make_spec

def make_spec(name):
    if name not in ("aki_lab_incident_tested", "aki_lab_incident_tested_nocreat"):
        return _make_spec_base(name)
    sp = _make_spec_base("aki_lab_incident")
    d = sp["data"]
    d = d[d[CREAT].notna()].reset_index(drop=True)
    num = [v for v in sp["num"] if not (name.endswith("_nocreat") and v == CREAT)]
    binc = [b for b in sp["bin"] if b in d.columns and d[b].nunique() > 1]
    return dict(name=name, data=d, num=num, bin=binc, cat=sp["cat"], label=sp["label"])

SENS_AKI = ["aki_lab_incident_tested", "aki_lab_incident_tested_nocreat"]
base = _make_spec_base("aki_lab_incident")["data"]
print(f"aki_lab_incident population: {len(base)} admissions, {int(base.y_aki.sum())} events "
      f"({100 * base.y_aki.mean():.2f}%)")
tested = base[CREAT].notna()
print(f"  creatinine measured in the first 24 h: {int(tested.sum())} admissions ({100 * tested.mean():.1f}%), "
      f"{int(base.loc[tested, 'y_aki'].sum())} events ({100 * base.loc[tested, 'y_aki'].mean():.2f}%)")
print(f"  not measured:                          {int((~tested).sum())} admissions, "
      f"{int(base.loc[~tested, 'y_aki'].sum())} events ({100 * base.loc[~tested, 'y_aki'].mean():.2f}%)")
for nm in SENS_AKI:
    sp = make_spec(nm)
    print(f"{nm:34s} admissions={len(sp['data'])}  events={int(sp['data'][sp['label']].sum())}  numeric={sp['num']}  binary={sp['bin']}")

RUN_SECTION_8 = True   # set to False to skip these two analyses when running section 9 only
if RUN_SECTION_8:
    for nm in SENS_AKI:
        ev = int(make_spec(nm)["data"]["y_aki"].sum())
        if ev < 40:
            print(f"WARNING: {nm} has only {ev} events; estimates will be unstable (report as exploratory).")
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

aki_lab_incident population: 19175 admissions, 202 events (1.05%)
  creatinine measured in the first 24 h: 3767 admissions (19.6%), 165 events (4.38%)
  not measured:                          15408 admissions, 37 events (0.24%)
aki_lab_incident_tested            admissions=3767  events=165  numeric=['age_at_admission', 'hemoglobin', 'creatinine']  binary=[]
aki_lab_incident_tested_nocreat    admissions=3767  events=165  numeric=['age_at_admission', 'hemoglobin']  binary=[]
[aki_lab_incident_tested] first outer fold took 0.5 min -> projected total about 7 min
[aki_lab_incident_tested] cross-validation finished in 7.1 min; every admission scored exactly once per repeat: True
  bootstrap: 5 replicates took 0 s -> projected 0.5 min for B=300
ANALYSIS aki_lab_incident_tested: 3767 admissions, 3275 patients, 165 events (4.38%), numeric=3, binary=0, categorical=5
Repeats=3, outer folds=5, inner folds=3, bootstrap B=300, QUICK=False

[0] Patients per outer partition (repeat 1 shown; every spli

### **9. Sensitivity of the SMM definition: admissions that are SMM only because of non-acute conditions**

> **Framework:** E9 · alternative outcome definition of stage E3 (non-acute codes removed).

The official CDC list includes a few codes for chronic or incidental conditions that are unlikely to be acute morbidity of the delivery:
nonruptured cerebral aneurysm (ICD-10 I67.1, ICD-9 437.3), carotid stenosis (I65.21, I65.22), amaurosis fugax (G45.3) and thoracic aortic
aneurysm without rupture (I71.2). The code-level audit (`sql/07_timing_audit/02_phenotype_audit_v3.sql`, query A2) found 14 SMM admissions
whose **only** qualifying codes are of this kind. This section removes those admissions from the population (they are not relabelled as
non-events) and re-runs two analyses:

| Analysis | Population |
|---|---|
| `excl_nonacute` | main population minus the 14 admissions (reference: `data/outputs_main.txt`) |
| `landmark_excl_nonacute` | `landmark` population minus those of the 14 admissions it contains (reference: `data/outputs_landmark.txt`) |

Requires `mimiciv.smm_code_hits_v3` (created by `02_phenotype_audit_v3.sql`). Reports: `data/outputs_excl_nonacute.txt` and
`data/outputs_landmark_excl_nonacute.txt`. To run only this section: set `RUN_MAIN_ANALYSES = False` (section 7) and `RUN_SECTION_8 = False`
(section 8), then run the notebook top to bottom. To skip it when running a later section, set `RUN_SECTION_9 = False`.


In [11]:
from sqlalchemy import text
NON_ACUTE_CODES = ("I671", "4373", "I6521", "I6522", "G453", "I712")
codes_sql = ", ".join(f"'{x}'" for x in NON_ACUTE_CODES)   # fixed constants, not user input
SQL_NON_ACUTE = text(f"""
    SELECT hadm_id
    FROM mimiciv.smm_code_hits_v3
    GROUP BY hadm_id
    HAVING BOOL_AND(icd_code IN ({codes_sql}))
""")
non_acute = set(pd.read_sql(SQL_NON_ACUTE, engine).hadm_id)   # kept in memory only

_make_spec_s8 = make_spec

def make_spec(name):
    if name not in ("excl_nonacute", "landmark_excl_nonacute"):
        return _make_spec_s8(name)
    sp = _make_spec_s8("main" if name == "excl_nonacute" else "landmark")
    d = sp["data"]
    d = d[~d.hadm_id.isin(non_acute)].reset_index(drop=True)
    return dict(name=name, data=d, num=sp["num"], bin=sp["bin"], cat=sp["cat"], label=sp["label"])

SENS_NONACUTE = ["excl_nonacute", "landmark_excl_nonacute"]
elig = df[df.eligible_24h & df.is_delivery_v2]
print(f"Admissions that are SMM only because of non-acute codes: {len(non_acute)} (expected 14 from query A2)  "
      f"[{'OK' if len(non_acute) == 14 else 'CHECK'}]")
print(f"  of which eligible SMM deliveries: {int(elig[elig.hadm_id.isin(non_acute)].y.sum())}")
for ref, nm in (("main", "excl_nonacute"), ("landmark", "landmark_excl_nonacute")):
    a, b = make_spec(ref)["data"], make_spec(nm)["data"]
    print(f"{nm:24s} admissions={len(b)} (reference {ref}: {len(a)})  events={int(b.y.sum())} (reference: {int(a.y.sum())})")

RUN_SECTION_9 = True
if RUN_SECTION_9:
    for nm in SENS_NONACUTE:
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

Admissions that are SMM only because of non-acute codes: 14 (expected 14 from query A2)  [OK]
  of which eligible SMM deliveries: 14
excl_nonacute            admissions=19369 (reference main: 19383)  events=431 (reference: 445)
landmark_excl_nonacute   admissions=19184 (reference landmark: 19198)  events=271 (reference: 285)
[excl_nonacute] first outer fold took 0.8 min -> projected total about 12 min
[excl_nonacute] cross-validation finished in 11.0 min; every admission scored exactly once per repeat: True
  bootstrap: 5 replicates took 1 s -> projected 1.2 min for B=300
ANALYSIS excl_nonacute: 19369 admissions, 13700 patients, 431 events (2.23%), numeric=3, binary=1, categorical=5
Repeats=3, outer folds=5, inner folds=3, bootstrap B=300, QUICK=False

[0] Patients per outer partition (repeat 1 shown; every split verified to share no patient between training and test)
 fold  train_adm  train_pat  train_events  train_event_pat  test_adm  test_pat  test_events  test_event_pat  shared_pat

### **10. Do richer predictors improve the prediction of SMM that is not yet present at 24 h?**

> **Framework:** Return R4 (back to E7: more predictors) · re-evaluated against criteria C1 and C2 for the second gate-P4 classification.

The main model uses only age, the administrative variables, hemoglobin and creatinine. With the full SMM label, adding the other laboratory
tests of the dictionary (role `laboratory sensitivity`, each with a missingness indicator) raised the AUROC from 0.79 to 0.82 and doubled
the AUPRC (`data/outputs_labs_extended.txt`), but that label mixes prediction with recognition of established morbidity. This section
repeats the comparison in the populations where SMM present at T has been removed:

| Analysis | Population | Predictors | Reference |
|---|---|---|---|
| `landmark_labs` | `landmark` | main + all `laboratory sensitivity` tests, each with a missingness indicator | `outputs_landmark.txt` |
| `landmark_labs_noind` | `landmark` | as `landmark_labs` but **no missingness indicators** (missing values imputed) | `landmark_labs` |
| `incident_documented_labs` | `incident_documented` | as `landmark_labs` | `outputs_incident_documented.txt` |
| `incident_strict_labs` | `incident_strict` | as `landmark_labs` | `outputs_incident_strict.txt` |

`landmark_labs_noind` separates the information in the laboratory **values** from the information in **which tests were ordered**
(the indicators). Ordering a test is legitimate information at 24 h (it reflects clinical concern); what matters is whether the gain
survives without it. The ICU-only variables are not used: the landmark population excludes ICU admission before T, so they are all missing.

To run only this section: `RUN_MAIN_ANALYSES = False`, `RUN_SECTION_8 = False`, `RUN_SECTION_9 = False`, then run the notebook top to bottom.
Reports: `data/outputs_<analysis>.txt` and `data/outputs_richer_predictors_summary.txt`.

In [12]:
_make_spec_s9 = make_spec
LAB10 = ["landmark_labs", "landmark_labs_noind", "incident_documented_labs", "incident_strict_labs"]

def make_spec(name):
    if name not in LAB10:
        return _make_spec_s9(name)
    base_name = name.replace("_labs_noind", "").replace("_labs", "")
    sp = _make_spec_s9(base_name)
    d = sp["data"]
    num = list(dict.fromkeys(sp["num"] + [v for v in VAR_LAB if v in d.columns]))
    if name.endswith("_noind"):
        binc = []
    else:
        binc = list(dict.fromkeys(sp["bin"] + [f"{v}_missing" for v in VAR_LAB if f"{v}_missing" in d.columns]))
        binc = [b for b in binc if d[b].nunique() > 1]
    return dict(name=name, data=d, num=num, bin=binc, cat=sp["cat"], label=sp["label"])

print("Laboratory variables added:", VAR_LAB)
for nm in LAB10:
    sp = make_spec(nm)
    print(f"{nm:26s} admissions={len(sp['data'])}  events={int(sp['data'][sp['label']].sum())}  "
          f"numeric={len(sp['num'])}  binary={len(sp['bin'])}  categorical={len(sp['cat'])}")

RUN_SECTION_10 = True
if RUN_SECTION_10:
    for nm in LAB10:
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

def richer_summary():
    rows = []
    for nm in LAB10:
        if nm not in results:
            continue
        pt, bt, y = results[nm]["point"], results[nm]["boot"], results[nm]["y"]
        for m in ("LR", "XGB", "Stack"):
            rows.append({"analysis": nm, "model": m, "events": int(y.sum()), "prevalence": f"{y.mean():.4f}",
                         "auroc": f"{pt[(m, 'auroc')]:.3f}" + _ci(bt, (m, "auroc")),
                         "auprc": f"{pt[(m, 'auprc')]:.3f}" + _ci(bt, (m, "auprc")),
                         "slope": f"{pt[(m, 'slope')]:.2f}", "O/E": f"{pt[(m, 'oe')]:.2f}"})
    print(f"RICHER PREDICTORS SUMMARY (QUICK={QUICK}); references: outputs_landmark, outputs_incident_documented, outputs_incident_strict")
    print(pd.DataFrame(rows).to_string(index=False))

if RUN_SECTION_10:
    emit("richer_predictors_summary", richer_summary)

Laboratory variables added: ['hematocrit', 'sodium', 'anion_gap', 'magnesium', 'chloride', 'lactate', 'phosphorus', 'glucose', 'bicarbonate', 'calcium_noionized', 'potassium']
landmark_labs              admissions=19198  events=285  numeric=14  binary=12  categorical=5
landmark_labs_noind        admissions=19198  events=285  numeric=14  binary=0  categorical=5
incident_documented_labs   admissions=19027  events=114  numeric=14  binary=12  categorical=5
incident_strict_labs       admissions=18996  events=83  numeric=14  binary=12  categorical=5
[landmark_labs] first outer fold took 1.1 min -> projected total about 17 min
[landmark_labs] cross-validation finished in 15.3 min; every admission scored exactly once per repeat: True
  bootstrap: 5 replicates took 1 s -> projected 1.1 min for B=300
ANALYSIS landmark_labs: 19198 admissions, 13584 patients, 285 events (1.48%), numeric=14, binary=12, categorical=5
Repeats=3, outer folds=5, inner folds=3, bootstrap B=300, QUICK=False

[0] Patients

### **11. Return R2: escalation of care after the prediction point**

> **Framework:** Return R2 (back to E3: an outcome dated after T) · criterion C2, and C1 through `escalation_landmark`.

The first gate-P4 classification (identifier, not predictor) left criterion C2 open: does the model predict an outcome **dated after T**?
This section fits the models on a timed outcome that does not depend on laboratory testing: **escalation of care after T**, defined in
`sql/07_timing_audit/04_escalation_label.sql` (table `mimiciv.escalation_v3`) as any of ICU admission, start of invasive mechanical
ventilation, a procedure-based SMM indicator, or in-hospital death after T. Admissions with ICU admission before T are excluded, because
they are already escalated at the prediction point (expected: 19,268 admissions, 101 events).

| Analysis | Population | Predictors | Label |
|---|---|---|---|
| `escalation` | eligible deliveries without ICU before T | main model | `y_esc` |
| `escalation_labs` | same | main model + all `laboratory sensitivity` tests with missingness indicators (R2 combined with R4) | `y_esc` |
| `escalation_landmark` | as `escalation`, also without SMM present or indeterminate at T | main model | `y_esc` |

`escalation_landmark` asks whether the signal survives once patients whose morbidity was already established at T are removed.
Criterion C2 is met if the AUROC is 0.75 or more **and** the lower 95% limit is 0.70 or more. With about 100 events the intervals will be
wide: the result is reported as exploratory if any analysis has fewer than 100 events.

Run `sql/07_timing_audit/04_escalation_label.sql` first. To run only this section: `RUN_MAIN_ANALYSES = False`, `RUN_SECTION_8 = False`,
`RUN_SECTION_9 = False`, `RUN_SECTION_10 = False`, `RUN_SECTION_12 = False`, then run the notebook top to bottom.
Reports: `data/outputs_<analysis>.txt` and `data/outputs_escalation_summary.txt`.

In [13]:
EXPECTED_ESC = {"population": 19268, "escalation": 101}   # Q-E1 of sql/07_timing_audit/03_escalation_counts.sql
esc = pd.read_sql("""SELECT hadm_id, icu_after, vent_after, smm_proc_after, death_after, escalation_any
                     FROM mimiciv.escalation_v3""", engine)
for c in ["icu_after", "vent_after", "smm_proc_after", "death_after", "escalation_any"]:
    esc[c] = esc[c].fillna(False).astype(bool)
chk = {"population": len(esc), "escalation": int(esc.escalation_any.sum())}
for k, v in chk.items():
    print(f"escalation {k}: {v}  (expected {EXPECTED_ESC[k]})  [{'OK' if v == EXPECTED_ESC[k] else 'CHECK: differs from Q-E1'}]")

df = df.drop(columns=[c for c in esc.columns if c != "hadm_id" and c in df.columns])
df = df.merge(esc, on="hadm_id", how="left")
df["in_esc_pop"] = df["escalation_any"].notna()
df["y_esc"] = df["escalation_any"].fillna(False).astype(int)

def _n(x):
    return str(int(x)) if x >= 10 else "<10"

_make_spec_s10b = make_spec
ESC11 = ["escalation", "escalation_labs", "escalation_landmark"]

def make_spec(name):
    if name not in ESC11:
        return _make_spec_s10b(name)
    sp = _make_spec_s10b("main")
    d = sp["data"]
    d = d[d.in_esc_pop & ~d.icu_in_24h]
    if name == "escalation_landmark":
        d = d[~d.smm_timing_class.isin(["1 prevalent", "2 indeterminate"])]
    d = d.reset_index(drop=True)
    num, binc = list(sp["num"]), list(sp["bin"])
    if name == "escalation_labs":
        num = list(dict.fromkeys(num + [v for v in VAR_LAB if v in d.columns]))
        binc = list(dict.fromkeys(binc + [f"{v}_missing" for v in VAR_LAB if f"{v}_missing" in d.columns]))
    binc = [b for b in binc if b in d.columns and d[b].nunique() > 1]
    return dict(name=name, data=d, num=num, bin=binc, cat=sp["cat"], label="y_esc")

for nm in ESC11:
    sp = make_spec(nm); d = sp["data"]
    print(f"{nm:20s} admissions={len(d):6d}  patients={d.subject_id.nunique():6d}  events={int(d.y_esc.sum()):4d}  "
          f"numeric={len(sp['num'])}  binary={len(sp['bin'])}  categorical={len(sp['cat'])}")

RUN_SECTION_11 = True
if RUN_SECTION_11:
    for nm in ESC11:
        ev = int(make_spec(nm)["data"]["y_esc"].sum())
        if ev < 100:
            print(f"NOTE: {nm} has {ev} events (< 100): report as exploratory.")
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

def escalation_summary():
    print(f"ESCALATION AFTER T (RETURN R2) SUMMARY (QUICK={QUICK}); bootstrap B={BOOT_SENS}; counts below 10 shown as <10")
    print("\n[1] Components of the composite (an admission can have several)")
    rows = []
    for nm in ESC11:
        d = make_spec(nm)["data"]; ev = d[d.y_esc == 1]
        rows.append({"analysis": nm, "admissions": _n(len(d)), "escalation": _n(len(ev)),
                     "icu_after": _n(ev.icu_after.sum()), "vent_after": _n(ev.vent_after.sum()),
                     "smm_proc_after": _n(ev.smm_proc_after.sum()), "death_after": _n(ev.death_after.sum()),
                     "with_smm_label": _n(ev.y.sum())})
    print(pd.DataFrame(rows).to_string(index=False))
    print("\n[2] Discrimination and calibration; C2 = AUROC >= 0.75 and lower 95% limit >= 0.70")
    rows = []
    for nm in ESC11:
        if nm not in results:
            continue
        pt, bt, y = results[nm]["point"], results[nm]["boot"], results[nm]["y"]
        for m in ("LR", "XGB", "Stack"):
            lo = np.nanpercentile(bt[(m, "auroc")], 2.5) if (m, "auroc") in bt else np.nan
            rows.append({"analysis": nm, "model": m, "events": int(y.sum()), "prevalence": f"{y.mean():.4f}",
                         "auroc": f"{pt[(m, 'auroc')]:.3f}" + _ci(bt, (m, "auroc")),
                         "auprc": f"{pt[(m, 'auprc')]:.3f}" + _ci(bt, (m, "auprc")),
                         "slope": f"{pt[(m, 'slope')]:.2f}", "O/E": f"{pt[(m, 'oe')]:.2f}",
                         "sens_top10": f"{pt[(m, 'sens_top10')]:.3f}",
                         "C2": "met" if (pt[(m, 'auroc')] >= 0.75 and lo >= 0.70) else "not met"})
    print(pd.DataFrame(rows).to_string(index=False))

if RUN_SECTION_11:
    emit("escalation_summary", escalation_summary)

escalation population: 19268  (expected 19268)  [OK]
escalation escalation: 101  (expected 101)  [OK]
escalation           admissions= 19268  patients= 13632  events= 101  numeric=3  binary=1  categorical=5
escalation_labs      admissions= 19268  patients= 13632  events= 101  numeric=14  binary=12  categorical=5
escalation_landmark  admissions= 19198  patients= 13584  events=  90  numeric=3  binary=1  categorical=5
[escalation] first outer fold took 0.9 min -> projected total about 13 min
[escalation] cross-validation finished in 12.4 min; every admission scored exactly once per repeat: True
  bootstrap: 5 replicates took 1 s -> projected 1.2 min for B=300
ANALYSIS escalation: 19268 admissions, 13632 patients, 101 events (0.52%), numeric=3, binary=1, categorical=5
Repeats=3, outer folds=5, inner folds=3, bootstrap B=300, QUICK=False

[0] Patients per outer partition (repeat 1 shown; every split verified to share no patient between training and test)
 fold  train_adm  train_pat  train_e

### **12. Race and ethnicity: model without the variable and performance by group**

> **Framework:** E7 decision on a predictor (race and ethnicity) and E10 subgroups and equity (performance and calibration by group).

The main model includes race and ethnicity (`ethnicity`) as recorded at registration. SMM rates differ by race and ethnicity, so
omitting the variable may miscalibrate the model for the groups at highest risk; but race in clinical algorithms can also
reproduce inequities. This section answers two questions with the same pipeline:

| Analysis | Population | Predictors | Reference |
|---|---|---|---|
| `main_norace` | `main` | main model **without** `ethnicity` | `outputs_main.txt` |
| `landmark_norace` | `landmark` | main model **without** `ethnicity` | `outputs_landmark.txt` |
| `race_subgroups` | `main` and `landmark`, with and without race | performance and calibration **by racial and ethnic group** | — |

Groups are collapsed from the MIMIC-IV categories: White, Black, Hispanic/Latina, Asian, Other, and Unknown or declined.
For each group the report gives admissions, events, AUROC, AUPRC, calibration-in-the-large, slope and O/E, with
patient-level bootstrap intervals for AUROC and O/E (`BOOT_SENS` replicates). A group with fewer than
`MIN_EVENTS_GROUP` events or non-events is reported with counts only. Counts below 10 are shown as `<10`.
The report also gives the paired difference in overall AUROC and AUPRC (with race minus without race, same bootstrap samples).

The question for the article is mainly **calibration by group**: does removing race change O/E in Black or Hispanic patients?

To run only this section: `RUN_MAIN_ANALYSES = False`, `RUN_SECTION_8 = False`, `RUN_SECTION_9 = False`, `RUN_SECTION_10 = False`, `RUN_SECTION_11 = False`,
then run the notebook top to bottom. `main` and `landmark` are run here if they are not already in `results`
(expect four cross-validations). Reports: `data/outputs_main_norace.txt`, `data/outputs_landmark_norace.txt` and
`data/outputs_race_subgroups.txt`.

In [14]:
RACE_COL = "ethnicity"
MIN_EVENTS_GROUP = 20
SUB_MODELS = ("LR", "XGB", "Stack")

def race_group(v):
    s = str(v).upper().strip() if pd.notna(v) else "UNKNOWN"
    if s.startswith("WHITE"):
        return "White"
    if s.startswith("BLACK"):
        return "Black"
    if s.startswith("HISPANIC") or s.startswith("SOUTH AMERICAN"):
        return "Hispanic/Latina"
    if s.startswith("ASIAN"):
        return "Asian"
    if s in ("UNKNOWN", "UNABLE TO OBTAIN", "PATIENT DECLINED TO ANSWER", "NONE", ""):
        return "Unknown or declined"
    return "Other"

def _n(x):
    return str(int(x)) if x >= 10 else "<10"

_make_spec_s10 = make_spec
NORACE = ["main_norace", "landmark_norace"]

def make_spec(name):
    if name not in NORACE:
        return _make_spec_s10(name)
    sp = _make_spec_s10(name.replace("_norace", ""))
    assert RACE_COL in sp["cat"], f"{RACE_COL} not among the categorical predictors: {sp['cat']}"
    return dict(sp, name=name, cat=[c for c in sp["cat"] if c != RACE_COL])

# Mapping of the recorded categories to the collapsed groups (eligible deliveries; aggregates only)
e = df[df.eligible_24h & df.is_delivery_v2]
mp = (e.assign(group=e[RACE_COL].map(race_group))
        .groupby(["group", RACE_COL], dropna=False).agg(admissions=("y", "size"), smm=("y", "sum")).reset_index())
mp["admissions"] = mp["admissions"].map(_n); mp["smm"] = mp["smm"].map(_n)
print("Recorded race and ethnicity categories and the collapsed group (counts below 10 suppressed):")
print(mp.to_string(index=False))
for nm in NORACE:
    sp = make_spec(nm)
    print(f"{nm:18s} admissions={len(sp['data'])}  events={int(sp['data'][sp['label']].sum())}  categorical={sp['cat']}")

def group_metrics(res, B, seed):
    """Metrics by collapsed race group, averaged over repeats; patient bootstrap within each group."""
    d = res["spec"]["data"]; y = res["y"].astype(float); g = res["g"]
    grp = d[RACE_COL].map(race_group).values
    rows, rng = [], np.random.default_rng(seed)
    for gr in ["White", "Black", "Hispanic/Latina", "Asian", "Other", "Unknown or declined"]:
        mk = grp == gr
        n, ev = int(mk.sum()), int(y[mk].sum())
        if n == 0:
            continue
        base = {"analysis": res["spec"]["name"], "group": gr, "admissions": _n(n), "events": _n(ev)}
        if ev < MIN_EVENTS_GROUP or (n - ev) < MIN_EVENTS_GROUP:
            rows.append(dict(base, model="-", note=f"fewer than {MIN_EVENTS_GROUP} events or non-events: counts only"))
            continue
        yg, gg = y[mk], g[mk]
        ug, inv = np.unique(gg, return_inverse=True)
        point = {m: pd.DataFrame([metrics_one(yg, res["preds"][m][r][mk], np.ones(n)) for r in range(N_REPEATS)]).mean()
                 for m in SUB_MODELS}
        boot = {m: {"auroc": [], "oe": []} for m in SUB_MODELS}
        for b in range(B):
            w = np.bincount(rng.integers(0, len(ug), len(ug)), minlength=len(ug))[inv].astype(float)
            if np.sum(w * yg) < 1 or np.sum(w * (1 - yg)) < 1:
                continue
            for m in SUB_MODELS:
                a, o = [], []
                for r in range(N_REPEATS):
                    pr = res["preds"][m][r][mk]
                    a.append(roc_auc_score(yg, pr, sample_weight=w))
                    o.append(np.sum(w * yg) / np.sum(w * pr))
                boot[m]["auroc"].append(np.mean(a)); boot[m]["oe"].append(np.mean(o))
        for m in SUB_MODELS:
            pt = point[m]
            ci = lambda k, dd: " ({:.{d}f}-{:.{d}f})".format(*np.nanpercentile(boot[m][k], [2.5, 97.5]), d=dd)
            rows.append(dict(base, model=m, prevalence=f"{ev / n:.4f}",
                             auroc=f"{pt['auroc']:.3f}" + ci("auroc", 3), auprc=f"{pt['auprc']:.3f}",
                             citl=f"{pt['citl']:.2f}", slope=f"{pt['slope']:.2f}",
                             oe=f"{pt['oe']:.2f}" + ci("oe", 2), note=""))
    return rows

def paired_race_diff(res_a, res_b, B, seed):
    """Overall AUROC/AUPRC with race (a) minus without race (b), same patient bootstrap samples."""
    assert np.array_equal(res_a["y"], res_b["y"]) and np.array_equal(res_a["g"], res_b["g"]), "rows not aligned"
    y = res_a["y"].astype(float); ug, inv = np.unique(res_a["g"], return_inverse=True)
    rng = np.random.default_rng(seed); rows = []
    for m in SUB_MODELS:
        dif = {"auroc": [], "auprc": []}
        for b in range(B):
            w = np.bincount(rng.integers(0, len(ug), len(ug)), minlength=len(ug))[inv].astype(float)
            if np.sum(w * y) < 1:
                continue
            for met, fn in (("auroc", roc_auc_score), ("auprc", average_precision_score)):
                va = np.mean([fn(y, res_a["preds"][m][r], sample_weight=w) for r in range(N_REPEATS)])
                vb = np.mean([fn(y, res_b["preds"][m][r], sample_weight=w) for r in range(N_REPEATS)])
                dif[met].append(va - vb)
        r0 = {"comparison": f"{res_a['spec']['name']} - {res_b['spec']['name']}", "model": m}
        for met in ("auroc", "auprc"):
            pa = res_a["point"][(m, met)]; pb = res_b["point"][(m, met)]
            lo, hi = np.nanpercentile(dif[met], [2.5, 97.5])
            r0[met] = f"{pa - pb:+.3f} ({lo:+.3f} to {hi:+.3f})"
        rows.append(r0)
    return rows

RUN_SECTION_12 = True
if RUN_SECTION_12:
    for nm in ["main", "landmark"]:
        if nm not in results:
            results[nm] = run_analysis(nm, BOOT_SENS)
            emit(nm + "_rerun12", lambda r=results[nm]: report(r))
    for nm in NORACE:
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

def race_summary():
    print(f"RACE AND ETHNICITY SUMMARY (QUICK={QUICK}); bootstrap B={BOOT_SENS}; groups with < {MIN_EVENTS_GROUP} events or non-events: counts only")
    print("\n[1] Overall discrimination, with race minus without race (paired patient bootstrap)")
    rows = []
    for a, b in (("main", "main_norace"), ("landmark", "landmark_norace")):
        if a in results and b in results:
            rows += paired_race_diff(results[a], results[b], BOOT_SENS, SEED + 120)
    print(pd.DataFrame(rows).to_string(index=False))
    print("\n[2] Performance and calibration by racial and ethnic group")
    rows = []
    for nm in ("main", "main_norace", "landmark", "landmark_norace"):
        if nm in results:
            rows += group_metrics(results[nm], BOOT_SENS, SEED + 121)
    print(pd.DataFrame(rows).fillna("").to_string(index=False))

if RUN_SECTION_12:
    emit("race_subgroups", race_summary)

Recorded race and ethnicity categories and the collapsed group (counts below 10 suppressed):
              group                                 ethnicity admissions smm
              Asian                                     ASIAN        842  22
              Asian                      ASIAN - ASIAN INDIAN        127 <10
              Asian                           ASIAN - CHINESE        661  17
              Asian                            ASIAN - KOREAN         56 <10
              Asian                  ASIAN - SOUTH EAST ASIAN        106 <10
              Black                             BLACK/AFRICAN        244 <10
              Black                    BLACK/AFRICAN AMERICAN       3396  92
              Black                        BLACK/CAPE VERDEAN        624  10
              Black                    BLACK/CARIBBEAN ISLAND        180 <10
    Hispanic/Latina                        HISPANIC OR LATINO        545 <10
    Hispanic/Latina        HISPANIC/LATINO - CENTRAL AMERICA

### **13. Criterion C3 for the extended laboratory model (return R4)**

> **Framework:** Return R4 · criterion C3 (detection bias) for the second gate-P4 classification.

Section 8 tested detection bias (criterion C3) for the main model. The second gate-P4 classification needs the same test for the model of
return R4, which adds all `laboratory sensitivity` tests. Laboratory AKI after T can only be detected when creatinine is measured again,
and ordering more tests in the first 24 h is itself a sign of clinical concern, so the extended model may gain discrimination simply from
**who was tested**.

| Analysis | Population | Predictors | Compare with |
|---|---|---|---|
| `aki_lab_incident_labs` | `aki_lab_incident` | main + all `laboratory sensitivity` tests, each with a missingness indicator | reference for this section |
| `aki_lab_incident_tested_labs` | `aki_lab_incident` restricted to admissions with creatinine measured in the first 24 h | as `aki_lab_incident_labs` | `aki_lab_incident_labs` |

C3 is met if the AUROC falls by 0.05 or less from `aki_lab_incident_labs` to `aki_lab_incident_tested_labs`.

To run only this section: set every other `RUN_...` switch to `False`, then run the notebook top to bottom.
Reports: `data/outputs_<analysis>.txt` and `data/outputs_c3_labs_summary.txt`.

In [15]:
_make_spec_s12 = make_spec
C3LABS = ["aki_lab_incident_labs", "aki_lab_incident_tested_labs"]

def make_spec(name):
    if name not in C3LABS:
        return _make_spec_s12(name)
    sp = _make_spec_s12("aki_lab_incident")
    d = sp["data"]
    if name == "aki_lab_incident_tested_labs":
        d = d[d[CREAT].notna()]
    d = d.reset_index(drop=True)
    num = list(dict.fromkeys(sp["num"] + [v for v in VAR_LAB if v in d.columns]))
    binc = list(dict.fromkeys(sp["bin"] + [f"{v}_missing" for v in VAR_LAB if f"{v}_missing" in d.columns]))
    binc = [b for b in binc if b in d.columns and d[b].nunique() > 1]
    return dict(name=name, data=d, num=num, bin=binc, cat=sp["cat"], label=sp["label"])

for nm in C3LABS:
    sp = make_spec(nm)
    print(f"{nm:30s} admissions={len(sp['data'])}  events={int(sp['data'][sp['label']].sum())}  "
          f"numeric={len(sp['num'])}  binary={len(sp['bin'])}  categorical={len(sp['cat'])}")

RUN_SECTION_13 = True
if RUN_SECTION_13:
    for nm in C3LABS:
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

def c3_labs_summary():
    print(f"CRITERION C3 FOR THE EXTENDED LABORATORY MODEL (QUICK={QUICK}); bootstrap B={BOOT_SENS}")
    rows = []
    for m in ("LR", "XGB", "Stack"):
        r = {"model": m}
        for nm in C3LABS:
            if nm in results:
                pt, bt = results[nm]["point"], results[nm]["boot"]
                r[f"{nm} events"] = int(results[nm]["y"].sum())
                r[f"{nm} auroc"] = f"{pt[(m, 'auroc')]:.3f}" + _ci(bt, (m, "auroc"))
        if all(nm in results for nm in C3LABS):
            drop = results[C3LABS[0]]["point"][(m, "auroc")] - results[C3LABS[1]]["point"][(m, "auroc")]
            r["auroc drop"] = f"{drop:+.3f}"
            r["C3"] = "met" if drop <= 0.05 else "not met"
        rows.append(r)
    print(pd.DataFrame(rows).to_string(index=False))

if RUN_SECTION_13:
    emit("c3_labs_summary", c3_labs_summary)

aki_lab_incident_labs          admissions=19175  events=202  numeric=14  binary=12  categorical=5
aki_lab_incident_tested_labs   admissions=3767  events=165  numeric=14  binary=10  categorical=5
[aki_lab_incident_labs] first outer fold took 1.1 min -> projected total about 17 min
[aki_lab_incident_labs] cross-validation finished in 14.1 min; every admission scored exactly once per repeat: True
  bootstrap: 5 replicates took 1 s -> projected 1.1 min for B=300
ANALYSIS aki_lab_incident_labs: 19175 admissions, 13565 patients, 202 events (1.05%), numeric=14, binary=12, categorical=5
Repeats=3, outer folds=5, inner folds=3, bootstrap B=300, QUICK=False

[0] Patients per outer partition (repeat 1 shown; every split verified to share no patient between training and test)
 fold  train_adm  train_pat  train_events  train_event_pat  test_adm  test_pat  test_events  test_event_pat  shared_patients
    1      15332      10848           156              156      3843      2717           46         

### **14. Gate P3 for the outcome of return R2 (escalation of care)**

> **Framework:** Return R2 sends the process back to stage E3 with a new outcome, so the new outcome must pass gate P3 again before its
> classification at gate P4 can stand. This section applies the criteria of Riley et al. to escalation of care, with the Cox-Snell R2
> observed in the cross-validated logistic regression (same method as section 7 of notebook 02).

For each escalation analysis it reports the number of candidate parameters `P` (columns after one-hot encoding), events per parameter,
the observed R2, the sample size and events required by each criterion, and the largest number of parameters the available data
support with expected shrinkage of at least 0.9. The main SMM analysis is added as a check against `outputs_main.txt`
(9,112 admissions and 209 events required).

The out-of-fold predictions are taken from `results` when the analyses of section 11 are still in memory. Otherwise the
cross-validation is re-run **without the bootstrap** (`run_cv` only, about 10-15 minutes per analysis).

To run only this section: set every `RUN_...` switch to `False`, run the notebook top to bottom.
Report: `data/outputs_p3_escalation.txt`.

In [16]:
def riley(P, phi, r2cs, shrink=0.9, deltas=(0.05, 0.01)):
    lnl0 = phi * math.log(phi) + (1 - phi) * math.log(1 - phi)
    max_r2 = 1 - math.exp(2 * lnl0)
    n1 = P / ((shrink - 1) * math.log(1 - r2cs / shrink))
    s_vh = r2cs / (r2cs + 0.05 * max_r2)
    n2 = P / ((s_vh - 1) * math.log(1 - r2cs / s_vh))
    out = {"max_r2cs": max_r2, "n_shrinkage": n1, "n_optimism": n2}
    for dl in deltas:
        out[f"n_risk_delta{dl}"] = (1.96 / dl) ** 2 * phi * (1 - phi)
    n = max(n1, n2, out[f"n_risk_delta{deltas[-1]}"])
    out["n_required"] = n
    out["events_required"] = n * phi
    out["epp_at_required"] = n * phi / P
    out["P_max_supported"] = None
    return out

def p_max(n, r2cs, shrink=0.9):
    """Largest number of parameters with expected shrinkage >= 0.9 for a given n and R2 (criterion 1 solved for P)."""
    return n * (shrink - 1) * math.log(1 - r2cs / shrink)

def riley_block(label, n, events, P, r2_list):
    phi = events / n
    print(f"\n{label}: n={n}, events={events}, prevalence={phi:.5f}, candidate parameters P={P}, EPP={events/P:.2f}")
    rows = []
    for tag, r2 in r2_list:
        rr = riley(P, phi, r2)
        rows.append({"R2_CS scenario": tag, "R2_CS": f"{r2:.4f}", "max R2_CS": f"{rr['max_r2cs']:.4f}",
                     "n shrinkage": f"{rr['n_shrinkage']:.0f}", "n optimism": f"{rr['n_optimism']:.0f}",
                     "n risk(+-0.05)": f"{rr['n_risk_delta0.05']:.0f}", "n risk(+-0.01)": f"{rr['n_risk_delta0.01']:.0f}",
                     "n required": f"{rr['n_required']:.0f}", "events required": f"{rr['events_required']:.0f}",
                     "available n": n, "available events": events,
                     "max P supported (n,R2)": f"{p_max(n, r2):.1f}"})
    print(pd.DataFrame(rows).to_string(index=False))

def count_parameters(spec):
    d = spec["data"]
    X = prep_frame(d, spec["num"], spec["bin"], spec["cat"])
    return int(make_prep(spec["num"], spec["bin"], spec["cat"]).fit(X).transform(X).shape[1])

def r2cs_from_oof(res):
    y = res["y"].astype(float); n = len(y); phi = y.mean()
    lnl0 = n * (phi * math.log(phi) + (1 - phi) * math.log(1 - phi))
    vals = []
    for r in range(N_REPEATS):
        p = np.clip(res["preds"]["LR"][r], 1e-9, 1 - 1e-9)
        lnl1 = float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))
        vals.append(max(1 - math.exp(2 / n * (lnl0 - lnl1)), 1e-6))
    return float(np.mean(vals))
P3_ANALYSES = ["main", "escalation", "escalation_labs", "escalation_landmark"]

RUN_SECTION_14 = True
p3_res = {}
if RUN_SECTION_14:
    for nm in P3_ANALYSES:
        if nm in results:
            p3_res[nm] = results[nm]
        else:
            print(f"[{nm}] not in memory: re-running the cross-validation without bootstrap")
            p3_res[nm] = run_cv(make_spec(nm))

def p3_report():
    print(f"GATE P3 (RILEY CRITERIA) FOR THE OUTCOME OF RETURN R2 (QUICK={QUICK})")
    print("R2_CS observed = mean over repeats of the Cox-Snell R2 of the out-of-fold logistic regression predictions")
    for nm in P3_ANALYSES:
        if nm not in p3_res:
            continue
        res = p3_res[nm]; sp = res["spec"]
        n, ev = len(res["y"]), int(res["y"].sum())
        P = count_parameters(sp)
        r2 = r2cs_from_oof(res)
        riley_block(f"{nm} (label {sp['label']})", n, ev, P, [("observed (cross-validated LR)", r2)])
        verdict = "passes" if riley(P, ev / n, r2)["n_required"] <= n else "does NOT pass"
        print(f"  -> gate P3 {verdict}: {n} admissions available")

if RUN_SECTION_14:
    emit("p3_escalation", p3_report)


GATE P3 (RILEY CRITERIA) FOR THE OUTCOME OF RETURN R2 (QUICK=False)
R2_CS observed = mean over repeats of the Cox-Snell R2 of the out-of-fold logistic regression predictions

main (label y): n=19383, events=445, prevalence=0.02296, candidate parameters P=32, EPP=13.91
               R2_CS scenario  R2_CS max R2_CS n shrinkage n optimism n risk(+-0.05) n risk(+-0.01) n required events required  available n  available events max P supported (n,R2)
observed (cross-validated LR) 0.0311    0.1964        9112       3191             34            862       9112             209        19383               445                   68.1
  -> gate P3 passes: 19383 admissions available

escalation (label y_esc): n=19268, events=101, prevalence=0.00524, candidate parameters P=32, EPP=3.16
               R2_CS scenario  R2_CS max R2_CS n shrinkage n optimism n risk(+-0.05) n risk(+-0.01) n required events required  available n  available events max P supported (n,R2)
observed (cross-validated LR) 0.0077